# Notebook 4: Batch Inference & Air Quality Forecast/Hindcast Graphs
**Author:** Guillén Concepción (Senior Data Scientist & MLOps Engineer)

This notebook downloads the registered XGBoost model from **Hopsworks Model Registry**, reads latest feature records, fetches 3-day meteorological forecast features, performs batch prediction, and generates **Air Quality Forecast/Hindcast graphs** for deployment on **GitHub Pages**.

In [ ]:
import os
import sys
import json
from pathlib import Path
from datetime import datetime, timezone

sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np
import plotly.graph_objects as go

from src.config import DEFAULT_LOCATION, MODEL_NAME, DATA_DIR
from src.data_fetcher import AirQualityDataFetcher, calculate_us_aqi_pm25
from src.features import generate_feature_pipeline, get_feature_names
from src.hopsworks_utils import FeatureStoreManager, ModelRegistryManager

## 1. Download Model from Hopsworks Model Registry

In [ ]:
fs_manager = FeatureStoreManager()
mr_manager = ModelRegistryManager(fs_manager)
model = mr_manager.load_model(MODEL_NAME)
print(f"Loaded trained XGBoost model: {type(model).__name__}")

## 2. Ingest Weather Forecast & Prepare Prediction Inputs

In [ ]:
fetcher = AirQualityDataFetcher(location_name=DEFAULT_LOCATION)
recent_df = fetcher.fetch_recent_data(past_days=14)
weather_forecast_df = fetcher.fetch_weather_forecast(forecast_days=3)

combined_df = pd.concat([recent_df, weather_forecast_df], ignore_index=True)
combined_df.drop_duplicates(subset=["timestamp"], keep="first", inplace=True)
combined_df.sort_values("timestamp", inplace=True)

featured_df = generate_feature_pipeline(combined_df)
feature_cols = get_feature_names(featured_df)

# Filter forecast horizon
forecast_df = featured_df.tail(72).copy()
X_forecast = forecast_df[feature_cols].ffill().fillna(0)

preds = np.clip(model.predict(X_forecast), a_min=0, a_max=None)
forecast_df["predicted_pm2_5"] = preds
forecast_df["predicted_us_aqi"] = forecast_df["predicted_pm2_5"].apply(calculate_us_aqi_pm25)
print(f"Generated 72-hour forecast predictions. Avg predicted PM2.5: {preds.mean():.2f} ug/m3")

## 3. Generate Forecast / Hindcast Graphs (Plotly)

In [ ]:
# Generate Hindcast (Past 48h Actual vs Predicted) & Forecast (Next 72h Predicted)
hindcast_df = featured_df.dropna(subset=["pm2_5"]).tail(48).copy()
hindcast_preds = np.clip(model.predict(hindcast_df[feature_cols].ffill().fillna(0)), a_min=0, a_max=None)
hindcast_df["predicted_pm2_5"] = hindcast_preds

fig = go.Figure()

# Ground Truth Hindcast
fig.add_trace(go.Scatter(
    x=hindcast_df["timestamp"],
    y=hindcast_df["pm2_5"],
    mode="lines+markers",
    name="Hindcast Observed PM2.5",
    line=dict(color="#10b981", width=2)
))

# Model Hindcast Predictions
fig.add_trace(go.Scatter(
    x=hindcast_df["timestamp"],
    y=hindcast_df["predicted_pm2_5"],
    mode="lines",
    name="Hindcast Model Fitted",
    line=dict(color="#f59e0b", width=2, dash="dash")
))

# 72h Future Forecast
fig.add_trace(go.Scatter(
    x=forecast_df["timestamp"],
    y=forecast_df["predicted_pm2_5"],
    mode="lines+markers",
    name="72h Future AQI Forecast",
    line=dict(color="#06b6d4", width=3)
))

# WHO Guideline
fig.add_trace(go.Scatter(
    x=pd.concat([hindcast_df["timestamp"], forecast_df["timestamp"]]),
    y=[15.0] * (len(hindcast_df) + len(forecast_df)),
    mode="lines",
    name="WHO 24h Safety Limit (15 ug/m3)",
    line=dict(color="#f43f5e", width=2, dash="dot")
))

fig.update_layout(
    title="Stockholm IoT Air Quality Sensor: Hindcast Evaluation & 72-Hour Forecast",
    xaxis_title="Timestamp (UTC)",
    yaxis_title="PM2.5 Concentration (ug/m3)",
    template="plotly_dark",
    height=500
)

# Save graph HTML artifact for GitHub Pages
output_html = Path.cwd().parent / "docs" / "air_quality_forecast_graph.html"
output_html.parent.mkdir(exist_ok=True)
fig.write_html(str(output_html))
print(f"✅ Air Quality Forecast/Hindcast Graph saved to {output_html} for GitHub Pages!")